# 03 · Cleaning and building the modelling table

**Steps per pond:** set sensor faults to NaN (`CLEANING_RANGES`; dissolved oxygen only at 0 or less, or above 20 mg/L) → remove IQR outliers → polynomial-regression imputation of each sensor from the other sensors (the best method in notebook 02), with imputed values kept within the range observed in that pond.

The imputer only sees sensor columns. Fish weight (the target) and fish length are never used to fill features, so no target information leaks into the inputs. Each pond is cleaned on its own, and evaluation later splits by pond, so imputation never crosses a train/test boundary.

**Output:** `data/processed/pond_clean.csv` (sensor rows) and `data/processed/pond_day.csv` (one row per pond per day, used for modelling).

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import importlib
import src.pond_pipeline
importlib.reload(src.pond_pipeline)  # pick up edits to src/ even if this kernel imported it earlier
from src.pond_pipeline import *  # noqa: F401,F403  (column names, thresholds, helpers)

# Raw pond CSVs live in data/raw/ (see data/README.md). Override with POND_RAW_DIR if needed.
RAW = Path(os.environ.get("POND_RAW_DIR", RAW_DIR))
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import pandas as pd

raw = load_all_ponds(RAW)
print("rows:", len(raw))
before = raw[MODEL_SENSOR_COLS].isna().mean().rename("missing before")

rows: 1114882


### Share of readings set to missing per pond (thesis Table 4)

After the range check and IQR outlier removal, before imputation.

In [3]:
flagged = pd.concat([remove_iqr_outliers(remove_invalid(g, MODEL_SENSOR_COLS), MODEL_SENSOR_COLS)
                     for _, g in raw.groupby("pond_id")])
missing_by_pond = (100 * flagged[MODEL_SENSOR_COLS].isna().groupby(flagged["pond_id"]).mean()).round(1)
missing_by_pond.to_csv(RESULTS_DIR / "missing_share_per_pond.csv")
missing_by_pond

,Temperature (C),Dissolved Oxygen(mg/L),PH,Nitrate(mg/L)
pond_id,,,,
1,3.1,22.7,0.5,0.0
2,0.4,24.5,46.3,0.0
3,2.5,31.9,21.0,0.0
4,2.7,12.8,9.9,0.0
6,0.1,12.9,32.1,0.1
7,2.8,31.4,45.8,0.0
8,1.1,39.2,44.7,0.3
9,1.7,27.9,47.5,0.0
10,16.8,55.8,96.5,17.7


In [4]:
clean = pd.concat([clean_pond(g) for _, g in raw.groupby("pond_id")], ignore_index=True)
after_range = remove_invalid(raw)[MODEL_SENSOR_COLS].isna().mean().rename("missing after range check")
pd.concat([before, after_range, clean[MODEL_SENSOR_COLS].isna().mean().rename("missing after imputation")], axis=1)

,missing before,missing after range check,missing after imputation
Temperature (C),0.0,0.000115,0.0
Dissolved Oxygen(mg/L),0.0,0.208905,0.0
PH,0.0,0.298302,0.0
Nitrate(mg/L),0.0,0.000174,0.0


Exact duplicate sensor rows add no information and are removed.

In [5]:
n = len(clean)
clean = clean.drop_duplicates(subset=["pond_id", "timestamp", *MODEL_SENSOR_COLS])
print(f"removed {n - len(clean)} duplicate rows")
clean[["pond_id", "timestamp", *MODEL_SENSOR_COLS, POPULATION, LENGTH, WEIGHT]].to_csv(
    PROCESSED_DIR / "pond_clean.csv", index=False)

removed 621 duplicate rows


## One row per pond per day

Sensors log every ~20 s, but fish weight is only measured on sampling days and repeated on every row in between. Modelling at row level would count the same weight thousands of times and let a random split put near-identical rows in both train and test. The modelling table therefore has one row per pond-day: the daily mean of each sensor, the fish weight and the days since the pond's first record.

In [6]:
pond_day = pond_day_table(clean)
pond_day.to_csv(PROCESSED_DIR / "pond_day.csv", index=False)
write_pipeline_version()
print("processed files written by", PIPELINE_VERSION)
print(pond_day.shape)
pond_day.groupby("pond_id").size().rename("days")

processed files written by 2026-11-25.4 (dates per column, stale weights, pond 11 restock, DO 0.01-20)
(856, 9)


pond_id
1      81
2     111
3      76
4      68
6      89
7     127
8      56
9     101
10     18
11    120
12      9
Name: days, dtype: int64

In [7]:
pond_day.describe().T

,count,mean,min,25%,50%,75%,max,std
pond_id,856.0,6.064252,1.0,3.0,7.0,9.0,12.0,3.355273
date,856,2021-09-01 06:31:57.757009408,2021-06-18 00:00:00,2021-07-19 00:00:00,2021-08-17 00:00:00,2021-10-03 00:00:00,2022-02-03 00:00:00,NaN
Temperature (C),856.0,25.063779,22.515625,24.173802,24.826179,25.719681,27.62362,1.209523
Dissolved Oxygen(mg/L),856.0,6.731707,0.014,4.420361,6.163451,8.416803,18.481501,3.604513
PH,856.0,7.304831,6.50437,7.050029,7.252487,7.534284,8.49266,0.404017
Nitrate(mg/L),856.0,7.904566,0.0,3.351733,5.509904,11.217835,22.683692,5.686704
Population,856.0,56.220794,50.0,50.0,50.0,50.0,75.0,10.814722
Fish_Weight(g),856.0,221.386524,2.869286,39.378214,115.305,306.1,1026.457143,246.522508
days_since_start,856.0,73.573598,0.0,30.0,59.0,106.25,223.0,55.453235
